Temporary script to rename files in folders with new slugifier.

In [ ]:
import pyrosm
import osmnx as ox
import networkx as nx
import csv
from growbikenet.functions import *
from growbikenet import constants
from growbikenet import settings

from slugify import slugify
def slugify_new(s): # Define temporarily until gbn 0.13.0 is released
    """Slugify a string
    
    Source: https://github.com/Chalarangelo/30-seconds-of-code/blob/master/content/snippets/python/s/slugify.md
    Note: A clean global solution would be using unidecode, but we do not want extra dependencies for this. We assume European city names in latin alphabet, some special letters like Hungarian long ö already mapped.

    Parameters
    ----------
    s : str
        String to slufigy

    Returns
    -------
    s : str
        Slugified string
    """
    s = s.lower().strip()
    s = re.sub(r'[\s-]+', '', s) # Remove white spaces, -
    s = re.sub(r'[^\w\s-]', '', s)
    s = re.sub(r'^-+|-+$', '', s)
    tab = str.maketrans(
        "áéíóúàèìùòâêîôûäëïöüǎěǐǒǔãẽĩõũăåæçčıłñňøœřßșşšůŷÿźž",
        "aeiouaeiouaeiouaeiouaeiouaeiouaaaccilnnoorssssuyyzz"
    )
    s = s.translate(tab)
    return s

In [ ]:
cities_path = "../cities/"
cityfilename = "european_capitalsand100000pop.csv"
folders = [cities_path+"cityexport/bike_networks/", cities_path+"cityexport/street_networks/", cities_path+"cityexport/pbfs/"]

In [ ]:
with open(cities_path+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {slugify(rows[0])+"_"+slugify(rows[3]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2], header[3]: rows[3], "boundaryfile": slugify(rows[0])+"_"+slugify(rows[3])} for rows in reader}

In [ ]:
for cityid, city_info in cities.items():
    for folder in folders:
        file = [entry for entry in os.listdir(folder) if entry.startswith(cityid)]
        if len(file) == 1: # rename
            filename = file[0]
            if folder.endswith("pbfs/"):
                ext = ".osm.pbf"
            else:
                ext = os.path.splitext(filename)[1]
            os.rename(folder+filename, folder+slugify_new(cityid)+ext)